# 05 - Comparación de modelos de detección de humo y fuego

Consolida los `metrics_summary.csv` de todos los experimentos y genera la tabla
y las figuras comparativas.

Este notebook **no necesita GPU, ni Drive, ni el dataset**: trabaja solo con los
CSV versionados en el repositorio, así que la comparación es reproducible por
cualquiera que clone el proyecto.

In [ ]:
# ============================================================
# Setup
# ============================================================

from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules
# Rama del repositorio desde la que se clona y a la que se commitean los
# resultados. Mientras el PR este abierto tiene que apuntar a la rama del PR;
# una vez mergeado, cambiar a "main".
REPO_BRANCH = "feat/modelos-adicionales-deteccion"

if IN_COLAB:
    !pip install -q -r https://raw.githubusercontent.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/{REPO_BRANCH}/requirements.txt
    !git clone -q -b {REPO_BRANCH} https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego.git /content/VpC2---Deteccion-de-humo-y-fuego
    PROJECT_DIR = Path("/content/VpC2---Deteccion-de-humo-y-fuego")
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)

In [ ]:
# ============================================================
# Carga de los resúmenes de cada experimento
# ============================================================

import pandas as pd

from src.reporting.summary import load_metrics_summaries

RESULTS_DIR = PROJECT_DIR / "reports" / "results"
df = load_metrics_summaries(RESULTS_DIR)

if df.empty:
    raise RuntimeError(
        f"No se encontró ningún metrics_summary.csv en {RESULTS_DIR}. "
        "Correr antes los notebooks 02, 03 y 04."
    )

# La comparación es entre tres familias arquitectónicas: si falta una, la tabla
# saldría igual pero diría algo distinto de lo que promete.
ESPERADOS = {"yolov8n_baseline", "fasterrcnn_r50fpn", "rtdetr_l"}
faltan = ESPERADOS - set(df["experiment"])
if faltan:
    raise RuntimeError(
        f"Faltan experimentos en {RESULTS_DIR}: {sorted(faltan)}. "
        "Correr los notebooks 02, 03 y 04 y commitear sus metrics_summary.csv."
    )

print(f"Experimentos encontrados: {len(df)}")
display(df)

> **Sobre `train_time_min`.** Los tres modelos entrenaron 30 épocas sobre una
> Tesla T4, así que la columna mide wall-clock comparable entre los tres. Aun así
> cubre un rango enorme —de 145 minutos a más de 18 horas—, porque mezcla el
> costo de la arquitectura con el del batch size que cada una tolera en la VRAM
> de la placa: 16 en YOLOv8n, 8 en RT-DETR y 4 en Faster R-CNN.
> Para comparar velocidad entre arquitecturas conviene mirar `fps`, que se mide
> en una sola pasada de validación, de a una imagen por vez y en la misma
> máquina que produjo el resto de la fila.


In [ ]:
# ============================================================
# Tabla comparativa principal
# ============================================================

COLUMNAS = [
    "experiment", "family", "params_M", "epochs", "train_time_min",
    "mAP50", "mAP50_95", "precision", "recall", "f1", "fps",
]

tabla = df[COLUMNAS].copy()
tabla.columns = [
    "Experimento", "Familia", "Params (M)", "Épocas", "Entrenamiento (min)",
    "mAP@0.5", "mAP@0.5:0.95", "Precisión", "Recall", "F1", "FPS",
]

display(tabla.style.background_gradient(subset=["mAP@0.5", "mAP@0.5:0.95"], cmap="Greens"))

In [ ]:
# ============================================================
# Desempeño por clase
# ============================================================

por_clase = df[
    ["experiment", "mAP50_smoke", "mAP50_fire", "mAP50_95_smoke", "mAP50_95_fire"]
].copy()
por_clase.columns = [
    "Experimento", "mAP@0.5 smoke", "mAP@0.5 fire",
    "mAP@0.5:0.95 smoke", "mAP@0.5:0.95 fire",
]

display(por_clase)

# El humo suele ser más difícil que el fuego: bordes difusos y sin forma definida.
diferencia = (df["mAP50_fire"] - df["mAP50_smoke"]).mean()
print(f"\nVentaja media de fire sobre smoke en mAP@0.5: {diferencia:+.4f}")

In [ ]:
# ============================================================
# Figuras comparativas
# ============================================================

from IPython.display import Image, display

from src.reporting.plots import plot_model_comparison

FIGURES_DIR = PROJECT_DIR / "reports" / "figures" / "comparacion"
rutas = plot_model_comparison(df, FIGURES_DIR)

for ruta in rutas:
    print(ruta.name)
    display(Image(filename=str(ruta)))

## Evolución a lo largo del entrenamiento

Las figuras anteriores comparan los valores finales; estas comparan la
trayectoria. Son las cuatro métricas de validación medidas al cierre de cada
época, tomadas del `results.csv` de cada experimento.

Las pérdidas quedan afuera a propósito: `box/cls/dfl` en YOLOv8n, `giou/cls/l1`
en RT-DETR y `loss_total` en Faster R-CNN son formulaciones distintas en escalas
distintas, así que superponerlas insinuaría una comparación que no significa
nada.

**Dos cosas que vale la pena mirar:**

- **RT-DETR salta entre la época 4 y la 5.** Son sus épocas de warmup: el LR sube
  de 3.3e-5 a 9.0e-5 y recién ahí el entrenamiento entra en régimen. Es la
  contracara de arrancar de una inicialización que todavía no predice nada útil:
  las tres primeras épocas quedan por debajo de 0.10 de mAP@0.5.
- **Faster R-CNN no termina en su mejor época.** Su pico es 0.781 en la época 25
  y cierra en 0.769, mientras la pérdida de entrenamiento sigue bajando: es
  sobreajuste sobre el final. La tabla reporta el 0.769 porque su bucle guarda
  los pesos de la última época y no los de la mejor, a diferencia del `best.pt`
  que Ultralytics elige por fitness para los otros dos.


In [ ]:
# ============================================================
# Evolución de las métricas por época
# ============================================================

from src.reporting.history import load_training_histories
from src.reporting.plots import plot_training_curves

historias = load_training_histories(RESULTS_DIR)

if historias.empty:
    raise RuntimeError(
        f"No se encontró ningún results.csv con métricas por época en {RESULTS_DIR}."
    )

print("Épocas por experimento:")
print(historias.groupby("experiment")["epoch"].max().to_string())

ruta = plot_training_curves(historias, FIGURES_DIR)
display(Image(filename=str(ruta)))


In [ ]:
# ============================================================
# Guardar la tabla consolidada
# ============================================================

COMPARISON_CSV = PROJECT_DIR / "reports" / "results" / "comparacion_modelos.csv"
df.to_csv(COMPARISON_CSV, index=False)

print("Tabla comparativa guardada en:", COMPARISON_CSV)

mejor = df.iloc[0]
print(
    f"\nMejor modelo por mAP@0.5: {mejor['experiment']} "
    f"({mejor['mAP50']:.4f}), a {mejor['fps']:.1f} FPS."
)

In [ ]:
# ============================================================
# Commit de la comparación
# ============================================================

import subprocess

%cd {PROJECT_DIR}

!git config user.name "Gabriela-Sol"
!git config user.email "solgab.salazar@gmail.com"

pull_result = subprocess.run(
    ["git", "pull", "--rebase", "origin", REPO_BRANCH], text=True, capture_output=True
)
print(pull_result.stdout, pull_result.stderr)

if pull_result.returncode != 0:
    raise RuntimeError("No se pudo completar git pull --rebase. Revisar conflictos.")

for path in ["reports/figures/comparacion/", "reports/results/comparacion_modelos.csv"]:
    if Path(path).exists():
        subprocess.run(["git", "add", path], check=True)
        print("Agregado:", path)

status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
print(status.stdout)

if status.stdout.strip():
    subprocess.run(["git", "commit", "-m", "results: comparacion entre modelos"], check=True)
    print(f"Commit creado. Para publicarlo: !git push origin {REPO_BRANCH}")
else:
    print("No hay cambios nuevos para commitear.")